# S07 · 01 — Drift real: enero contra julio

**Objetivo.** Medir drift sobre datos reales de la NYC TLC, con las particiones fijas
del curso, y llegar a la pregunta que no tiene respuesta automática:
*¿esto es drift que exige reentrenar, o es estacionalidad que conviene modelar?*

**Requisitos.** Haber corrido `uv run taxi data` al menos una vez (descarga y cachea
las particiones). Todo lo demás sale de `data/processed/`.

**Ruta del notebook.**

1. Cargar la referencia (2023-01..03) y dos periodos de "producción": 2023-07 y 2024-01.
2. Comparar distribuciones a ojo, con gráficas.
3. Calcular KS, chi-cuadrado, PSI y Jensen-Shannon **a mano** con `scipy`.
4. Ver por qué `p < 0.05` falla, variando `n` sobre los mismos datos.
5. Repetir el análisis con Evidently 0.7 y comparar los dos motores.
6. Decidir: ¿drift o estacionalidad? La evidencia se construye, no se copia.
7. Medir la degradación real del RMSE del modelo de enero sobre julio.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

from taxi import config
from taxi.features import contract as fc
from taxi.models import train
from taxi.monitoring import estadistico as est
from taxi.monitoring import reporte

print("referencia :", [p.etiqueta for p in config.PARTICIONES_TRAIN])
print("produccion :", [p.etiqueta for p in config.PARTICIONES_PRODUCCION])
print("umbral de columnas con drift:", config.UMBRAL_DRIFT_COLUMNAS)

## 1. Cargar los datos reales

`train.cargar_split` usa el cache de `data/processed/` si ya está materializado. La
primera vez descarga de la TLC; a partir de ahí es lectura de parquet.

Fíjate en qué se carga: **no hay ni una línea de `np.random`**. Es la diferencia
entre practicar y jugar.

In [ ]:
referencia = train.cargar_split(list(config.PARTICIONES_TRAIN))
julio = train.cargar_particion(config.Particion(2023, 7))
enero_siguiente = train.cargar_particion(config.Particion(2024, 1))

for nombre, df in [
    ("referencia 2023-01..03", referencia),
    ("produccion 2023-07", julio),
    ("produccion 2024-01", enero_siguiente),
]:
    print(f"{nombre:24s} {len(df):>8,} filas")

COLUMNAS_NUM = [*fc.FEATURES_NUMERICAS, fc.TARGET_REGRESION]
COLUMNAS_CAT = list(fc.FEATURES_CATEGORICAS)
print("\nnumericas :", COLUMNAS_NUM)
print("categoricas:", COLUMNAS_CAT)

## 2. Mirar antes de medir

Un test estadístico no sustituye mirar los datos. Este paso existe porque los
resúmenes numéricos esconden cambios de forma: dos distribuciones pueden tener la
misma media y ser distintas en todo lo demás.

In [ ]:
resumen = pd.DataFrame(
    {
        "ref_2023Q1": referencia[COLUMNAS_NUM].mean(),
        "jul_2023": julio[COLUMNAS_NUM].mean(),
        "ene_2024": enero_siguiente[COLUMNAS_NUM].mean(),
    }
)
resumen["delta_jul_%"] = 100 * (resumen["jul_2023"] / resumen["ref_2023Q1"] - 1)
resumen["delta_ene24_%"] = 100 * (resumen["ene_2024"] / resumen["ref_2023Q1"] - 1)
resumen.round(3)

In [ ]:
# Histogramas y CDF empiricas. La CDF es la que hay que mirar para el KS: el
# estadistico D es literalmente la maxima separacion vertical entre las dos curvas.
fig, ejes = plt.subplots(2, 2, figsize=(13, 8))

for eje, columna in zip(ejes[0], ["trip_distance", fc.TARGET_REGRESION]):
    limite = referencia[columna].quantile(0.99)
    bins = np.linspace(0, limite, 60)
    eje.hist(referencia[columna], bins=bins, density=True, alpha=0.55, label="ref 2023Q1")
    eje.hist(julio[columna], bins=bins, density=True, alpha=0.55, label="jul 2023")
    eje.set_title(f"{columna} — densidad")
    eje.legend()

for eje, columna in zip(ejes[1], ["trip_distance", fc.TARGET_REGRESION]):
    for etiqueta, serie in [
        ("ref 2023Q1", referencia[columna]),
        ("jul 2023", julio[columna]),
        ("ene 2024", enero_siguiente[columna]),
    ]:
        x = np.sort(serie.to_numpy())
        eje.plot(x, np.arange(1, len(x) + 1) / len(x), label=etiqueta)
    eje.set_xlim(0, referencia[columna].quantile(0.99))
    eje.set_title(f"{columna} — CDF empirica")
    eje.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Las categoricas: top de zonas de origen, en proporcion. Ojo con las categorias
# NUEVAS, que son las que un test categorico diluye entre las demas celdas.
top = referencia["PULocationID"].value_counts(normalize=True).head(12).index
comparacion = pd.DataFrame(
    {
        "ref_2023Q1": referencia["PULocationID"].value_counts(normalize=True).reindex(top),
        "jul_2023": julio["PULocationID"].value_counts(normalize=True).reindex(top),
        "ene_2024": enero_siguiente["PULocationID"].value_counts(normalize=True).reindex(top),
    }
).fillna(0.0)
comparacion.plot.bar(figsize=(12, 4), title="PULocationID — proporcion de las 12 zonas mas frecuentes")
plt.tight_layout()
plt.show()

nuevas = set(enero_siguiente["PULocationID"]) - set(referencia["PULocationID"])
print("zonas presentes en 2024-01 y ausentes en la referencia:", sorted(nuevas))

## 3. KS, chi-cuadrado, PSI y Jensen-Shannon a mano

Ahora sin librería de monitoreo: solo `scipy`. El objetivo es que quede claro qué
hace un detector de drift por dentro, cuál es su binning y dónde se toman las
decisiones.

In [ ]:
from scipy import stats

filas = []
for columna in COLUMNAS_NUM:
    ref, act = referencia[columna], julio[columna]
    resultado = stats.ks_2samp(ref.to_numpy(), act.to_numpy())
    filas.append(
        {
            "columna": columna,
            "test": "KS",
            "estadistico": resultado.statistic,
            "p_valor": resultado.pvalue,
            "efecto (D o V)": resultado.statistic,
            "psi": est.psi(ref, act),
            "js": est.distancia_jensen_shannon(ref, act),
        }
    )

for columna in COLUMNAS_CAT:
    ref, act = referencia[columna], julio[columna]
    # Se reutiliza el helper del modulo: agrupa categorias raras para que las
    # frecuencias esperadas del chi-cuadrado sean validas.
    parcial = est.evaluar_categorica(ref, act, columna=columna)
    filas.append(
        {
            "columna": columna,
            "test": "chi2",
            "estadistico": parcial.estadistico,
            "p_valor": parcial.p_valor,
            "efecto (D o V)": parcial.tamano_efecto,
            "psi": parcial.psi,
            "js": parcial.jensen_shannon,
        }
    )

pd.DataFrame(filas).set_index("columna").round(5)

**Lee la columna `p_valor`.** Con este volumen de datos casi todos son
indistinguibles de cero. Si el criterio fuera `p < 0.05`, la conclusión sería "drift
en todo", y eso no te dice qué hacer.

Compara ahora con la columna de efecto: ahí sí hay orden de magnitud entre columnas.

In [ ]:
# El veredicto del modulo, con las tres politicas, sobre los mismos datos.
for criterio in est.CRITERIOS:
    resultado = est.detectar_drift(
        referencia,
        julio,
        columnas_numericas=COLUMNAS_NUM,
        columnas_categoricas=COLUMNAS_CAT,
        criterio=criterio,
        umbral_columnas=config.UMBRAL_DRIFT_COLUMNAS,
    )
    marcadas = [c.columna for c in resultado.con_drift]
    print(f"criterio={criterio:8s} -> {len(marcadas)}/{len(resultado.columnas)} columnas: {marcadas}")

## 4. Por qué `p < 0.05` falla: el mismo cambio, distinta n

Experimento controlado sobre los datos reales: se toma **la misma diferencia** entre
enero y julio y se submuestrean tamaños crecientes. El tamaño de efecto (`D`) se
mantiene estable —es una propiedad de las distribuciones— y el p-valor se derrumba,
que es una propiedad del tamaño de muestra.

In [ ]:
columna = "trip_distance"
rng = np.random.default_rng(config.SEMILLA)
tamanos = [200, 1_000, 5_000, 20_000, 50_000, min(len(referencia), len(julio))]

filas = []
for n in tamanos:
    a = referencia[columna].sample(n=n, random_state=config.SEMILLA).to_numpy()
    b = julio[columna].sample(n=n, random_state=config.SEMILLA).to_numpy()
    r = stats.ks_2samp(a, b)
    filas.append(
        {
            "n": n,
            "D (efecto)": r.statistic,
            "p_valor": r.pvalue,
            "D critico 5%": 1.36 * np.sqrt(2 / n),
            "alerta si p<0.05": r.pvalue < 0.05,
            "alerta si D>=umbral": r.statistic >= est.umbral_de(columna, est.TIPO_NUMERICA),
        }
    )
pd.DataFrame(filas).set_index("n").round(6)

In [ ]:
# Control negativo del experimento: dos mitades de la MISMA particion.
# Si el detector marcara drift aqui, el detector estaria roto.
mitad_a = referencia.sample(frac=0.5, random_state=1)
mitad_b = referencia.drop(mitad_a.index)

control = est.detectar_drift(
    mitad_a,
    mitad_b,
    columnas_numericas=COLUMNAS_NUM,
    columnas_categoricas=COLUMNAS_CAT,
)
print(reporte.resumen(control))

El control negativo es el test que más gente olvida y el único que demuestra que el
detector informa algo en lugar de solo alertar.

Ojo con lo que **no** demuestra: bajo la hipótesis nula verdadera, el criterio por
p-valor también se porta bien (su tasa de error es alfa, por construcción). Corre la
celda siguiente y compruébalo. El problema del p-valor no está en el nulo: está en
las alternativas ciertas pero triviales, que es lo que midió el barrido de `n` del
paso 4.

Y el control negativo sirve para algo más útil todavía: **calibrar los umbrales**. El
efecto medido entre dos mitades de la referencia es el ruido del instrumento. Un
umbral por debajo de ese ruido produce falsos positivos garantizados.

In [ ]:
control_malo = est.detectar_drift(
    mitad_a,
    mitad_b,
    columnas_numericas=COLUMNAS_NUM,
    columnas_categoricas=COLUMNAS_CAT,
    criterio="p_valor",
)
print("falsas alarmas con criterio p_valor sobre el nulo:", [c.columna for c in control_malo.con_drift])

In [ ]:
# Calibracion: ruido del instrumento vs umbral configurado vs senal observada.
base = est.linea_base_nula(
    referencia,
    columnas_numericas=COLUMNAS_NUM,
    columnas_categoricas=COLUMNAS_CAT,
)


def efectos(df_actual: pd.DataFrame) -> dict:
    resultado = est.detectar_drift(
        referencia,
        df_actual,
        columnas_numericas=COLUMNAS_NUM,
        columnas_categoricas=COLUMNAS_CAT,
    )
    return {c.columna: c.tamano_efecto for c in resultado.columnas}


calibracion = pd.DataFrame(
    {
        "ruido (nulo)": pd.Series(base),
        "umbral configurado": pd.Series(
            {
                c: est.umbral_de(
                    c, est.TIPO_NUMERICA if c in COLUMNAS_NUM else est.TIPO_CATEGORICA
                )
                for c in base
            }
        ),
        "senal vs 2023-07": pd.Series(efectos(julio)),
        "senal vs 2024-01": pd.Series(efectos(enero_siguiente)),
    }
).round(4)
calibracion["umbral > ruido?"] = calibracion["umbral configurado"] > calibracion["ruido (nulo)"]
calibracion

Mira la fila de `PU_DO`. El par origen-destino tiene miles de niveles, y la V de
Cramér tiene **sesgo positivo** cuando hay muchas celdas con conteos bajos: sobre dos
mitades del mismo mes marca un efecto de ~0.12, aunque ahí no pasa absolutamente
nada. Un umbral de 0.10 para esa columna —que es el default razonable para una
categórica— habría alertado sobre datos idénticos.

Por eso `UMBRALES_POR_FEATURE` lleva 0.15 para `PU_DO`: **el umbral se fijó después
de medir el ruido, no antes**. Ese orden es la diferencia entre un umbral defendible
y un número copiado de un blog. Está documentado en
[ADR 003](../../../docs/adr/003-umbrales-de-drift.md).

## 5. El mismo análisis con Evidently 0.7

La API cambió por completo respecto a la de los tutoriales que circulan. Ver la tabla
de migración en el README de la sesión (§4.2). Lo esencial:

- `from evidently import DataDefinition, Dataset, Report`
- `Dataset.from_pandas(df, data_definition=esquema)`
- `report.run(actual, referencia)` — **primero el actual**
- `snapshot.dict()`, no `as_dict()`

In [ ]:
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

esquema = DataDefinition(numerical_columns=COLUMNAS_NUM, categorical_columns=COLUMNAS_CAT)
columnas = COLUMNAS_NUM + COLUMNAS_CAT

ds_ref = Dataset.from_pandas(referencia[columnas], data_definition=esquema)
ds_act = Dataset.from_pandas(julio[columnas], data_definition=esquema)

report = Report([DataDriftPreset(), DataSummaryPreset()], include_tests=True)
evaluacion = report.run(ds_act, ds_ref)  # (current, reference)

destino = config.REPORTS_DIR / "notebook_drift_2023Q1_vs_2023-07.html"
destino.parent.mkdir(parents=True, exist_ok=True)
evaluacion.save_html(str(destino))
print("HTML en:", destino.relative_to(config.PROJECT_ROOT))

In [ ]:
bruto = evaluacion.dict()
print("claves del dict:", list(bruto.keys()))

# Asi se ve una metrica por columna. El `value` NO siempre es un p-valor: depende
# del `method`. Ver la tabla del README (seccion 4.3).
for metrica in bruto["metrics"][:4]:
    print(metrica["metric_name"], "->", metrica["value"])

In [ ]:
# La traduccion a la estructura del curso vive en UNA funcion. Si Evidently cambia
# el formato otra vez, se toca solo ese archivo.
resultado_evidently = reporte.desde_evidently(
    bruto,
    columnas_numericas=COLUMNAS_NUM,
    columnas_categoricas=COLUMNAS_CAT,
    umbral_columnas=config.UMBRAL_DRIFT_COLUMNAS,
)
print(reporte.resumen(resultado_evidently))

In [ ]:
# Y el check completo, que es lo que corre el CI: HTML + JSON + exit code.
from taxi.monitoring import check_drift as cd

resultado_check = cd.ejecutar_check(
    actual=(config.Particion(2023, 7),),
    mlflow_tracking=False,  # ponlo en True si tienes el servidor de MLflow arriba
)
print(cd.formatear(resultado_check))
print("\nexit code:", resultado_check.codigo_salida)

## 6. La pregunta difícil: ¿drift o estacionalidad?

Aquí termina lo mecánico y empieza el juicio profesional. El detector dice "las
distribuciones cambiaron". No dice qué hacer, y las dos lecturas posibles llevan a
acciones opuestas:

| Lectura | Qué implica | Acción |
|---|---|---|
| **Drift genuino y permanente** | el mundo cambió y no va a volver | reentrenar con datos recientes |
| **Estacionalidad recurrente** | el mundo cambia todos los julios y vuelve | modelar el patrón; reentrenar no lo arregla |

Reentrenar en respuesta a un patrón estacional es perseguir la propia cola: el modelo
se ajusta a julio, llega septiembre y vuelve a "driftear". Peor: cada reentrenamiento
es un cambio en producción, con su riesgo de regresión.

**Evidencia que distingue las dos hipótesis.** Si el cambio fuera estacional, dos
eneros deberían parecerse más entre sí que un enero y un julio. Eso es comprobable
con las particiones que ya tienes.

In [ ]:
def efecto_por_columna(df_a: pd.DataFrame, df_b: pd.DataFrame) -> pd.Series:
    """Tamano de efecto por columna entre dos periodos."""
    resultado = est.detectar_drift(
        df_a,
        df_b,
        columnas_numericas=COLUMNAS_NUM,
        columnas_categoricas=COLUMNAS_CAT,
    )
    return pd.Series({c.columna: c.tamano_efecto for c in resultado.columnas})


comparativa = pd.DataFrame(
    {
        "2023Q1 vs 2023-07 (otro mes)": efecto_por_columna(referencia, julio),
        "2023Q1 vs 2024-01 (mismo mes, +1 anio)": efecto_por_columna(referencia, enero_siguiente),
    }
).round(4)
comparativa["mas_grande"] = np.where(
    comparativa.iloc[:, 0] > comparativa.iloc[:, 1], "cambio de mes", "paso del tiempo"
)
comparativa

**Cómo se lee esta tabla.**

- Si el efecto contra julio es mucho mayor que contra enero de 2024, el cambio es
  **estacional**: la posición en el año pesa más que el paso del tiempo.
- Si el efecto contra 2024-01 es comparable o mayor, hay un **cambio de tendencia**
  además de la estacionalidad, y ahí sí hay razón para revisar el modelo.
- Puede pasar (y suele pasar) que la respuesta sea **distinta por feature**. Eso es
  información valiosa: dice qué features hay que modelar y cuáles hay que vigilar.

Ninguna de las tres conclusiones sale de un umbral. Sale de mirar la tabla y de saber
que en NYC hay verano.

**Ejercicio de decisión (escríbelo antes de continuar):**

1. Con la tabla anterior, ¿tu diagnóstico es estacionalidad, tendencia o mezcla?
2. Si es estacionalidad, ¿qué feature añadirías? El contrato ya tiene `hora_pickup` y
   `dia_semana_pickup`, pero **no** mes ni indicador de temporada. ¿Basta con `mes`?
   ¿Cuántos ciclos de historia necesitas en el train para que el modelo pueda
   aprenderlo?
3. ¿Qué evidencia adicional pedirías que **no** esté en estos datos? (Pista: 2022-07
   y 2024-07 existen en el portal de la TLC. Dos julios distintos convierten una
   hipótesis en una serie.)
4. Coste de equivocarse en cada dirección: ¿qué pasa si reentrenas y era
   estacionalidad? ¿Y si no reentrenas y era un cambio permanente?

In [ ]:
# Opcional (requiere red): traer un julio mas para convertir la hipotesis en serie.
# Descomenta si tienes conexion. Dos julios y dos eneros son mejor evidencia que uno.
#
# julio_2024 = train.cargar_particion(config.Particion(2024, 7))
# print(efecto_por_columna(julio, julio_2024).round(4))  # julio vs julio: deberia ser bajo

## 7. La prueba que zanja la discusión: ¿se degradó el modelo?

Todo lo anterior mide **entradas**. Lo que importa es la calidad de las
**predicciones**. En este caso guía tenemos el lujo de tener las etiquetas de julio
(son datos históricos), así que podemos medir la degradación real. En producción,
esto solo es posible después del label lag.

Se entrena con la referencia y se evalúa en tres periodos. Mide y compara los números
que salgan; no memorices los de nadie.

Antes de ejecutar, **escribe tu predicción**: ¿cuánto peor esperas que sea el RMSE de
julio? ¿Y el de enero de 2024, un año después? Compararla con el resultado es la parte
que enseña.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

pipeline = train.pipeline_lineal()
pipeline.fit(referencia[fc.FEATURES], referencia[fc.TARGET_REGRESION])

periodos = {
    "valid 2023-04 (mes siguiente)": train.cargar_valid(),
    "prod 2023-07 (verano)": julio,
    "prod 2024-01 (un anio despues)": enero_siguiente,
}

filas = []
for nombre, df in periodos.items():
    y_real = df[fc.TARGET_REGRESION]
    y_pred = pipeline.predict(df[fc.FEATURES])
    filas.append(
        {
            "periodo": nombre,
            "n": len(df),
            "rmse": root_mean_squared_error(y_real, y_pred),
            "mae": mean_absolute_error(y_real, y_pred),
            "sesgo (pred - real)": float(np.mean(y_pred - y_real)),
        }
    )

degradacion = pd.DataFrame(filas).set_index("periodo")
base = degradacion.loc["valid 2023-04 (mes siguiente)", "rmse"]
degradacion["rmse vs valid %"] = 100 * (degradacion["rmse"] / base - 1)
degradacion.round(4)

In [ ]:
# Degradacion por subgrupo. El promedio esconde regresiones locales: es el mismo
# argumento del gate de promocion de S06, aplicado ahora al monitoreo.
from taxi.models import evaluate

for nombre, df in [("2023-04 (valid)", train.cargar_valid()), ("2023-07", julio), ("2024-01", enero_siguiente)]:
    y_pred = pipeline.predict(df[fc.FEATURES])
    por_subgrupo = evaluate.metricas_por_subgrupo(df, df[fc.TARGET_REGRESION], y_pred)
    solo_rmse = {k: round(v, 3) for k, v in por_subgrupo.items() if k.startswith("rmse_")}
    print(f"\n=== {nombre} ===")
    for clave, valor in sorted(solo_rmse.items()):
        print(f"  {clave:22s} {valor}")

## 8. Cierre

Lo que quedó demostrado, en orden:

1. Hay drift real entre enero y julio, y no hizo falta inventarlo. También quedó claro
   que el drift real es **mucho más sutil** que el sintético: con las particiones del
   curso, el veredicto de dataset puede quedar por debajo del umbral aunque columnas
   concretas se muevan. Eso es realista, y es incómodo, que es el punto.
2. El p-valor no distingue "cambió" de "cambió lo suficiente". El tamaño de efecto sí.
3. El control negativo (dos mitades de la misma partición) valida el detector y además
   **calibra los umbrales**. Sin él, un umbral es una superstición.
4. La misma señal admite dos diagnósticos opuestos, y la evidencia para elegir se
   construye comparando periodos comparables.
5. La degradación de RMSE es la que justifica actuar, y solo está disponible después
   del label lag. Con label lag alto hay que decidir con menos información: ahí es
   donde la política de reentrenamiento escrita vale más que cualquier gráfica.

**La conclusión que suele sorprender.** Si en tu ejecución el RMSE de julio empeora
poco y el de enero de 2024 no empeora, tienes delante el caso que este curso quiere
que reconozcas: **hay drift de datos medible y no hay degradación de performance**. La
respuesta correcta ahí no es reentrenar. Es registrar el hallazgo, ajustar los
umbrales con la evidencia y seguir vigilando. Reentrenar "porque el detector se puso
rojo" es exactamente el reflejo que el monitoreo mal entendido produce.

**Siguiente:** `02-observabilidad-del-servicio.ipynb` — la otra mitad del monitoreo,
la que mide el servicio y no los datos.